# Thermal Agent v2 — TD3 + Dynamic Degradation Coupling

## Improvements over v1

| Issue in v1 | Fix in v2 |
|---|---|
| Q_REF=global mean → penalty never fires (Q_pred always >Q_REF) | **Dynamic Q_REF**: per-episode surrogate query at t=0 |
| Vanilla DDPG: Q overestimation, unstable | **TD3**: twin critics, delayed actor, target smoothing |
| OU noise constant throughout training | **Noise decay**: sigma 0.3→0.05 over 1500 episodes |
| State dim=6, no capacity trend | **State dim=7**: adds ΔQ (capacity gradient) feature |
| 500 episodes (under-trained) | **1500 episodes** |
| Ablation: 100 eps each → all same | **300 eps each** → differentiates β values |
| No LR scheduling | **ReduceLROnPlateau** for actor and critic |
| 20-ep final eval | **50-ep final eval** (tighter error bars) |


## 1 · Imports

In [1]:
import numpy as np, torch, torch.nn as nn
import torch.nn.functional as F, torch.optim as optim
import pickle, os, copy, random, json
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import deque
import warnings; warnings.filterwarnings('ignore')

SEED=42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
torch.backends.cudnn.deterministic=True

BASE_DIR    = r'/Users/mahizhan/Documents/Sem7/Github/data-driven-prediction-of-battery-cycle-life-before-capacity-degradation'
MODEL_DIR   = os.path.join(BASE_DIR,'Model')
RESULTS_DIR = os.path.join(BASE_DIR,'thermal_agent_v2_results')
os.makedirs(RESULTS_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device} | PyTorch {torch.__version__}')


Device: cpu | PyTorch 2.14.0


## 2 · Load Surrogate

In [2]:
class DegradationSurrogate1DCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv=nn.Sequential(
            nn.Conv1d(3,32,5,padding=2),nn.BatchNorm1d(32),nn.ReLU(),nn.MaxPool1d(2),
            nn.Conv1d(32,64,3,padding=1),nn.BatchNorm1d(64),nn.ReLU(),nn.MaxPool1d(2),
            nn.Conv1d(64,128,3,padding=1),nn.BatchNorm1d(128),nn.ReLU(),nn.AdaptiveAvgPool1d(4),
        )
        self.head=nn.Sequential(
            nn.Flatten(),
            nn.Linear(512,256),nn.ReLU(),nn.Dropout(0.3),
            nn.Linear(256,128),nn.ReLU(),nn.Dropout(0.2),
            nn.Linear(128,1),
        )
    def forward(self,x): return self.head(self.conv(x))

ckpt      = torch.load(os.path.join(MODEL_DIR,'degradation_surrogate_1dcnn.pth'), map_location=device)
surrogate = DegradationSurrogate1DCNN().to(device)
surrogate.load_state_dict(ckpt['model_state'])
surrogate.eval()

CH_MEAN = np.array(ckpt['ch_mean'],dtype=np.float32)
CH_STD  = np.array(ckpt['ch_std'], dtype=np.float32)
Y_MEAN  = float(ckpt['y_mean'])
Y_STD   = float(ckpt['y_std'])
NUM_PTS = int(ckpt['num_pts'])

def query_surrogate(V_tr, I_tr, T_tr):
    n=len(V_tr)
    if n<2: return Y_MEAN
    g_raw=np.linspace(0,1,n); g_fix=np.linspace(0,1,NUM_PTS)
    V_f=np.interp(g_fix,g_raw,V_tr)
    I_f=np.interp(g_fix,g_raw,I_tr)
    T_f=np.interp(g_fix,g_raw,T_tr)
    X=np.stack([V_f,I_f,T_f],axis=0)[np.newaxis]
    X=(X-CH_MEAN)/CH_STD
    with torch.no_grad():
        y=surrogate(torch.FloatTensor(X).to(device)).item()
    return float(np.clip(y*Y_STD+Y_MEAN, Y_MEAN-4*Y_STD, Y_MEAN+2*Y_STD))

print(f'Surrogate ready | y_mean={Y_MEAN:.4f} Ah')


Surrogate ready | y_mean=1.0331 Ah


## 3 · Load Data & Build Profiles

In [3]:
print('Loading batches...')
batch1=pickle.load(open(os.path.join(BASE_DIR,'batch1.pkl'),'rb'))
for k in ['b1c8','b1c10','b1c12','b1c13','b1c22']: del batch1[k]
batch2=pickle.load(open(os.path.join(BASE_DIR,'batch2.pkl'),'rb'))
b2k=['b2c7','b2c8','b2c9','b2c15','b2c16']
b1k=['b1c0','b1c1','b1c2','b1c3','b1c4']
add_len=[662,981,1060,208,482]
for i,bk in enumerate(b1k):
    batch1[bk]['cycle_life']+=add_len[i]
    for j in batch1[bk]['summary']:
        if j=='cycle':
            batch1[bk]['summary'][j]=np.hstack((batch1[bk]['summary'][j],
                batch2[b2k[i]]['summary'][j]+len(batch1[bk]['summary'][j])))
        else:
            batch1[bk]['summary'][j]=np.hstack((batch1[bk]['summary'][j],
                batch2[b2k[i]]['summary'][j]))
    lc=len(batch1[bk]['cycles'])
    for j,jk in enumerate(batch2[b2k[i]]['cycles']):
        batch1[bk]['cycles'][str(lc+j)]=batch2[b2k[i]]['cycles'][jk]
for k in b2k: del batch2[k]
batch3=pickle.load(open(os.path.join(BASE_DIR,'batch3.pkl'),'rb'))
for k in ['b3c37','b3c2','b3c23','b3c32','b3c42','b3c43']: del batch3[k]
numBat1,numBat2,numBat3=len(batch1),len(batch2),len(batch3)
numBat=numBat1+numBat2+numBat3
bat_dict={**batch1,**batch2,**batch3}
all_keys=list(bat_dict.keys())
train_ind=np.arange(1,numBat1+numBat2-1,2)
train_keys=[all_keys[i] for i in train_ind if i<len(all_keys)]
profiles=[]
for key in train_keys:
    cell=bat_dict[key]
    cl=int(cell['cycle_life'])
    for c_str,c_data in cell['cycles'].items():
        c_num=int(c_str)
        if c_num<10: continue
        try:
            I=np.array(c_data['I'],dtype=np.float32)
            V=np.array(c_data['V'],dtype=np.float32)
            if len(I)>=20: profiles.append({'I':I,'V':V,'cycle_num':c_num,'cycle_life':cl})
        except: pass
print(f'Profiles: {len(profiles):,} | Cells: {len(train_keys)}')


Loading batches...


Profiles: 27,770 | Cells: 41


## 4 · BatteryThermalEnv v2

**Key fix — Dynamic Q_REF:**

At `reset()`, the surrogate is queried with the first 5 datapoints of the selected
cycle to establish `Q_ep_ref` — the baseline capacity for this specific cell/cycle.
The degradation penalty is then `max(0, Q_ep_ref − Q_pred)`, which fires whenever
temperature damages capacity relative to the episode-start baseline.

**State (dim=7):** `[T/50, T_prev/50, I/6, cycle_norm, SOC, Q_pred/Q_ep_ref, ΔQ_norm]`

The new `ΔQ_norm` feature gives the agent an instantaneous signal of how fast
capacity is changing — critical for proactive (not reactive) thermal management.


In [4]:
class BatteryThermalEnvV2:
    STATE_DIM=7; ACTION_DIM=1

    def __init__(self, profiles, alpha=0.1, beta=5.0, gamma_e=0.01,
                 T_target=30.0, T_max=45.0, ambient=25.0, dt=1.0, steps=300):
        self.profiles=profiles; self.alpha=alpha; self.beta=beta
        self.gamma_e=gamma_e; self.T_target=T_target; self.T_max=T_max
        self.ambient=ambient; self.C_th=800.0; self.R_int=0.02
        self.h_conv=5.0; self.P_max=50.0; self.dt=dt; self.steps_per_ep=steps
        self.Q_ep_ref=Y_MEAN; self.Q_prev=Y_MEAN

    def reset(self):
        self.T=self.T_prev=self.ambient+np.random.uniform(-2,2)
        self.step_idx=0; self.SOC=1.0
        self.V_tr=[]; self.I_tr=[]; self.T_tr=[]
        p=random.choice(self.profiles)
        self.I_profile=p['I']; self.V_profile=p['V']
        self.cycle_num=p['cycle_num']; self.cycle_life=max(p['cycle_life'],1)
        # ── Dynamic Q_REF: query surrogate at episode start (FIX) ───────────
        n0=min(5,len(self.V_profile))
        Vref=list(self.V_profile[:n0])
        Iref=list(abs(self.I_profile[:n0]))
        Tref=[self.T]*n0
        self.Q_ep_ref=query_surrogate(Vref,Iref,Tref)
        self.Q_prev=self.Q_ep_ref
        return self._obs()

    def _I(self):
        return float(abs(self.I_profile[min(self.step_idx,len(self.I_profile)-1)]))

    def _obs(self):
        I=self._I()
        q=query_surrogate(self.V_tr,self.I_tr,self.T_tr) if self.V_tr else self.Q_ep_ref
        dQ=(q-self.Q_prev)/(self.Q_ep_ref+1e-8)
        return np.array([
            self.T/50.0, self.T_prev/50.0, I/6.0,
            self.cycle_num/self.cycle_life, self.SOC,
            q/(self.Q_ep_ref+1e-8),   # ratio to episode-start capacity
            float(np.clip(dQ,-1,1)),  # capacity gradient (new feature)
        ],dtype=np.float32)

    def step(self, action):
        action=float(np.clip(action,-1,1))
        P_cool=(action+1)/2*self.P_max
        I=self._I()
        Q_gen=I**2*self.R_int; Q_loss=self.h_conv*(self.T-self.ambient)
        dT=(Q_gen-P_cool-Q_loss)/self.C_th*self.dt
        self.T_prev=self.T
        self.T=float(np.clip(self.T+dT,self.ambient-5,80))
        vi=min(self.step_idx,len(self.V_profile)-1)
        self.V_tr.append(float(self.V_profile[vi]))
        self.I_tr.append(I); self.T_tr.append(self.T)
        self.SOC=float(np.clip(self.SOC-I*self.dt/3600/(self.Q_ep_ref+1e-8),0,1))
        q_pred=query_surrogate(self.V_tr,self.I_tr,self.T_tr)
        # ── Reward: now fires because Q_ep_ref > Q_pred when T damages capacity
        deg_penalty=max(0.0, self.Q_ep_ref-q_pred)   # KEY FIX
        safety=self.T>self.T_max
        reward=(-self.alpha*(self.T-self.T_target)**2
                -self.beta*deg_penalty
                -self.gamma_e*(P_cool/self.P_max)
                +(-1000.0 if safety else 0.0))
        self.Q_prev=q_pred
        self.step_idx+=1
        done=self.step_idx>=self.steps_per_ep or safety
        info=dict(T=self.T,P_cool=P_cool,q_pred=q_pred,
                  deg_penalty=deg_penalty,safety=safety,
                  r_temp=-self.alpha*(self.T-self.T_target)**2,
                  r_degrad=-self.beta*deg_penalty,
                  Q_ep_ref=self.Q_ep_ref)
        return self._obs(),reward,done,info

print('BatteryThermalEnvV2 ready | State dim:', BatteryThermalEnvV2.STATE_DIM)


BatteryThermalEnvV2 ready | State dim: 7


## 5 · TD3 — Twin-Delayed Deep Deterministic Policy Gradient

TD3 fixes three key failure modes of vanilla DDPG:

| DDPG failure | TD3 fix |
|---|---|
| Q-value overestimation biases actor | Twin critics — take `min(Q1, Q2)` for target |
| Actor updates too frequently, amplifying critic errors | Delayed actor update (every 2 critic steps) |
| Deterministic target policy — exploits critic errors | Target policy smoothing — add clipped noise to target action |

**Reference:** Fujimoto et al. (2018), *Addressing Function Approximation Error in Actor-Critic Methods*, ICML.


In [5]:
class ActorTD3(nn.Module):
    def __init__(self,sd=7,ad=1):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(sd,256),nn.LayerNorm(256),nn.ReLU(),
            nn.Linear(256,256),nn.LayerNorm(256),nn.ReLU(),
            nn.Linear(256,ad),nn.Tanh()
        )
        nn.init.uniform_(self.net[-2].weight,-3e-3,3e-3)
        nn.init.uniform_(self.net[-2].bias,-3e-3,3e-3)
    def forward(self,s): return self.net(s)


class CriticTD3(nn.Module):
    """Twin critics (Q1, Q2) — both share same architecture."""
    def __init__(self,sd=7,ad=1):
        super().__init__()
        self.Q1=nn.Sequential(
            nn.Linear(sd+ad,256),nn.ReLU(),
            nn.Linear(256,256),nn.ReLU(),nn.Linear(256,1))
        self.Q2=nn.Sequential(
            nn.Linear(sd+ad,256),nn.ReLU(),
            nn.Linear(256,256),nn.ReLU(),nn.Linear(256,1))
        for m in [self.Q1[-1],self.Q2[-1]]:
            nn.init.uniform_(m.weight,-3e-3,3e-3)
            nn.init.uniform_(m.bias,-3e-3,3e-3)
    def forward(self,s,a):
        sa=torch.cat([s,a],dim=-1)
        return self.Q1(sa),self.Q2(sa)
    def Q1_only(self,s,a):
        return self.Q1(torch.cat([s,a],dim=-1))


class ReplayBuffer:
    def __init__(self,cap=200_000): self.buf=deque(maxlen=cap)
    def push(self,s,a,r,ns,d): self.buf.append((s,float(a),float(r),ns,float(d)))
    def sample(self,bs):
        b=random.sample(self.buf,bs)
        s,a,r,ns,d=zip(*b)
        return(torch.FloatTensor(np.array(s)).to(device),
               torch.FloatTensor(np.array(a)).unsqueeze(-1).to(device),
               torch.FloatTensor(np.array(r)).unsqueeze(-1).to(device),
               torch.FloatTensor(np.array(ns)).to(device),
               torch.FloatTensor(np.array(d)).unsqueeze(-1).to(device))
    def __len__(self): return len(self.buf)


class TD3Agent:
    def __init__(self,sd=7,ad=1,alr=1e-4,clr=3e-4,
                 gamma=0.99,tau=0.005,buf_cap=200_000,bs=256,
                 policy_delay=2,noise_clip=0.5,target_noise=0.2):
        self.actor=ActorTD3(sd,ad).to(device)
        self.critic=CriticTD3(sd,ad).to(device)
        self.at=copy.deepcopy(self.actor); self.ct=copy.deepcopy(self.critic)
        for p in self.at.parameters(): p.requires_grad_(False)
        for p in self.ct.parameters(): p.requires_grad_(False)
        self.ao=optim.Adam(self.actor.parameters(),lr=alr)
        self.co=optim.Adam(self.critic.parameters(),lr=clr)
        # LR scheduling
        self.as_=optim.lr_scheduler.ReduceLROnPlateau(self.ao,'min',factor=0.5,patience=50)
        self.cs_=optim.lr_scheduler.ReduceLROnPlateau(self.co,'min',factor=0.5,patience=50)
        self.buf=ReplayBuffer(buf_cap)
        self.bs=bs; self.gamma=gamma; self.tau=tau
        self.policy_delay=policy_delay; self.noise_clip=noise_clip
        self.target_noise=target_noise; self._update_cnt=0
        # Exploration noise (decays over training)
        self.noise_sigma=0.3

    def act(self, state, explore=True):
        s=torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad(): a=self.actor(s).cpu().numpy()[0]
        if explore:
            a+=np.random.normal(0,self.noise_sigma,size=a.shape)
        return float(np.clip(a,-1,1))

    def update(self):
        if len(self.buf)<self.bs: return 0.0,0.0
        s,a,r,ns,d=self.buf.sample(self.bs)
        self._update_cnt+=1
        # ── Critic update ────────────────────────────────────────────────────
        with torch.no_grad():
            noise=torch.clamp(
                torch.randn_like(a)*self.target_noise,
                -self.noise_clip,self.noise_clip)
            na=torch.clamp(self.at(ns)+noise,-1,1)
            tQ1,tQ2=self.ct(ns,na)
            tQ=r+self.gamma*(1-d)*torch.min(tQ1,tQ2)   # take min (TD3 fix)
        cQ1,cQ2=self.critic(s,a)
        cL=F.mse_loss(cQ1,tQ)+F.mse_loss(cQ2,tQ)
        self.co.zero_grad(); cL.backward()
        torch.nn.utils.clip_grad_norm_(self.critic.parameters(),1.0)
        self.co.step()
        # ── Delayed actor update ─────────────────────────────────────────────
        aL=0.0
        if self._update_cnt%self.policy_delay==0:
            aL=-self.critic.Q1_only(s,self.actor(s)).mean()
            self.ao.zero_grad(); aL.backward()
            torch.nn.utils.clip_grad_norm_(self.actor.parameters(),0.5)
            self.ao.step()
            aL=float(aL)
            # Soft target updates
            for p,tp in zip(self.actor.parameters(),self.at.parameters()):
                tp.data.copy_(self.tau*p.data+(1-self.tau)*tp.data)
            for p,tp in zip(self.critic.parameters(),self.ct.parameters()):
                tp.data.copy_(self.tau*p.data+(1-self.tau)*tp.data)
        return float(cL),aL

    def decay_noise(self, ep, total_eps, sigma_start=0.3, sigma_end=0.05):
        """Linear noise decay over training."""
        self.noise_sigma=sigma_end+(sigma_start-sigma_end)*(1-ep/total_eps)

    def save(self,path):
        torch.save({'actor':self.actor.state_dict(),'critic':self.critic.state_dict()},path)
        print(f'Saved: {path}')


class PIDController:
    def __init__(self,T_target=30.0,Kp=0.05,Ki=0.005,Kd=0.001):
        self.T_target=T_target; self.Kp=Kp; self.Ki=Ki; self.Kd=Kd
        self.integral=self.prev_err=0.0
    def reset(self): self.integral=self.prev_err=0.0
    def act(self,state,explore=None):
        T=state[0]*50.0; e=T-self.T_target
        self.integral=float(np.clip(self.integral+e,-100,100))
        d=e-self.prev_err; self.prev_err=e
        return float(np.clip(self.Kp*e+self.Ki*self.integral+self.Kd*d,-1,1))

print('TD3Agent and PIDController ready.')


TD3Agent and PIDController ready.


## 6 · Training — 1500 Episodes with Noise Decay

Two parallel agents are trained:
- **TD3-Surrogate (β=5)**: full degradation coupling — the proposed method
- **TD3-NoSurrogate (β=0)**: thermal-only ablation baseline (no degradation coupling)

Both use identical TD3 hyperparameters, only β differs.


In [6]:
EPISODES=1500; STEPS=300; BS=256
ALPHA=0.1; BETA=5.0; GAMMA_E=0.01; EVAL_EVERY=100

# Main agent (beta=5)
env_main  = BatteryThermalEnvV2(profiles,alpha=ALPHA,beta=BETA,gamma_e=GAMMA_E,steps=STEPS)
agent_main= TD3Agent(sd=7,bs=BS)

# No-surrogate baseline (beta=0)
env_ab0   = BatteryThermalEnvV2(profiles,alpha=ALPHA,beta=0.0,gamma_e=GAMMA_E,steps=STEPS)
agent_ab0 = TD3Agent(sd=7,bs=BS)

pid = PIDController()

def run_ep(ag, env_obj, explore=True, is_pid=False):
    state=env_obj.reset()
    if is_pid: ag.reset()
    tot_r=0.0; t_errs=[]; q_preds=[]; deg_ps=[]; p_cools=[]; safes=0
    cL_sum=aL_sum=n_upd=0
    for _ in range(env_obj.steps_per_ep):
        a=ag.act(state,explore=explore) if not is_pid else ag.act(state)
        ns,r,done,info=env_obj.step(a)
        if not is_pid and explore:
            ag.buf.push(state,a,r,ns,float(done))
            cL,aL=ag.update(); cL_sum+=cL; aL_sum+=aL; n_upd+=1
        tot_r+=r; t_errs.append(abs(info['T']-env_obj.T_target))
        q_preds.append(info['q_pred'])
        deg_ps.append(info['deg_penalty']); p_cools.append(info['P_cool'])
        if info['safety']: safes+=1
        state=ns
        if done: break
    Qref=env_obj.Q_ep_ref
    return {'R':tot_r,'T_MAE':float(np.mean(t_errs)),
            'cap_pct':float(np.mean(q_preds)/Qref*100) if Qref>0 else 100.0,
            'deg_penalty':float(np.mean(deg_ps)),
            'cooling':float(np.sum(p_cools)),'safety':safes,
            'cL':cL_sum/(n_upd+1e-8),'aL':aL_sum/(n_upd+1e-8)}

train_R_main=[]; train_R_ab0=[]
eval_main=[]; eval_ab0=[]; eval_pid_=[]

print('='*70)
print(f'Training TD3-Surrogate (beta=5) + TD3-NoSurrogate (beta=0) | {EPISODES} eps')
print('='*70)

for ep in range(EPISODES):
    agent_main.decay_noise(ep,EPISODES)
    agent_ab0.decay_noise(ep,EPISODES)
    m_main=run_ep(agent_main,env_main,explore=True)
    m_ab0 =run_ep(agent_ab0, env_ab0, explore=True)
    train_R_main.append(m_main['R']); train_R_ab0.append(m_ab0['R'])
    if (ep+1)%EVAL_EVERY==0:
        em=run_ep(agent_main,env_main,explore=False)
        ea=run_ep(agent_ab0, env_ab0, explore=False)
        ep_=run_ep(pid, env_main, explore=False, is_pid=True)
        eval_main.append(em); eval_ab0.append(ea); eval_pid_.append(ep_)
        print(f"Ep{ep+1:5d} | sigma={agent_main.noise_sigma:.3f} | "
              f"TD3-Surr: R={em['R']:7.1f} T_MAE={em['T_MAE']:.2f} "
              f"Cap={em['cap_pct']:.1f}% DegPen={em['deg_penalty']:.5f} | "
              f"PID: R={ep_['R']:7.1f} T_MAE={ep_['T_MAE']:.2f}")

    # LR step every 100 eps
    if (ep+1)%100==0:
        agent_main.as_.step(m_main['R'])
        agent_main.cs_.step(m_main['R'])

agent_main.save(os.path.join(MODEL_DIR,'thermal_agent_td3_v2.pth'))
print('Training complete.')


Training TD3-Surrogate (beta=5) + TD3-NoSurrogate (beta=0) | 1500 eps


Ep  100 | sigma=0.283 | TD3-Surr: R= -706.1 T_MAE=4.84 Cap=103.2% DegPen=0.00024 | PID: R=-1219.6 T_MAE=6.36


Ep  200 | sigma=0.267 | TD3-Surr: R= -874.3 T_MAE=5.37 Cap=103.3% DegPen=0.00023 | PID: R= -836.1 T_MAE=5.26


Ep  300 | sigma=0.250 | TD3-Surr: R= -791.9 T_MAE=5.13 Cap=103.3% DegPen=0.00023 | PID: R= -883.8 T_MAE=5.41


Ep  400 | sigma=0.233 | TD3-Surr: R= -549.8 T_MAE=4.25 Cap=103.2% DegPen=0.00023 | PID: R=-1006.1 T_MAE=5.79


...

Ep 1100 | sigma=0.117 | TD3-Surr: R= -777.1 T_MAE=5.08 Cap=101.8% DegPen=0.00187 | PID: R=-1178.3 T_MAE=6.26


Ep 1200 | sigma=0.100 | TD3-Surr: R= -573.5 T_MAE=4.35 Cap=103.8% DegPen=0.00020 | PID: R= -819.7 T_MAE=5.20


Ep 1300 | sigma=0.084 | TD3-Surr: R= -777.7 T_MAE=5.08 Cap=101.4% DegPen=0.00196 | PID: R=-1052.4 T_MAE=5.92


Ep 1400 | sigma=0.067 | TD3-Surr: R= -781.8 T_MAE=5.09 Cap=101.7% DegPen=0.00233 | PID: R= -924.4 T_MAE=5.54


Ep 1500 | sigma=0.050 | TD3-Surr: R= -992.4 T_MAE=5.74 Cap=101.7% DegPen=0.00080 | PID: R=-1010.1 T_MAE=5.80
Saved: /Users/mahizhan/Documents/Sem7/Github/data-driven-prediction-of-battery-cycle-life-before-capacity-degradation/Model/thermal_agent_td3_v2.pth
Training complete.


## 7 · β Ablation — 300 Episodes per β Value

**β ∈ {0, 0.5, 1, 2, 5, 10, 20}** — 300 training episodes each.
300 episodes gives TD3 enough time to learn a meaningfully different policy for each β.

This produces the core analytical contribution: the **Pareto frontier** between
thermal accuracy and capacity preservation — unique to this paper.


In [7]:
BETA_VALS=[0.0, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0]
ablation={}
print(f'Running beta ablation: {len(BETA_VALS)} values x 300 eps each...')

for bv in BETA_VALS:
    torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
    env_ab=BatteryThermalEnvV2(profiles,alpha=ALPHA,beta=bv,gamma_e=GAMMA_E,steps=STEPS)
    ag_ab=TD3Agent(sd=7,bs=BS)
    for ep in range(300):
        ag_ab.decay_noise(ep,300)
        run_ep(ag_ab,env_ab,explore=True)
    tm_s,cp_s,dp_s=[],[],[]
    for _ in range(20):
        m=run_ep(ag_ab,env_ab,explore=False)
        tm_s.append(m['T_MAE']); cp_s.append(m['cap_pct']); dp_s.append(m['deg_penalty'])
    ablation[bv]={
        'T_MAE':float(np.mean(tm_s)),'T_MAE_std':float(np.std(tm_s)),
        'cap_pct':float(np.mean(cp_s)),'cap_pct_std':float(np.std(cp_s)),
        'deg_penalty':float(np.mean(dp_s))
    }
    print(f'  beta={bv:5.1f}: T_MAE={ablation[bv]["T_MAE"]:.3f}+/-{ablation[bv]["T_MAE_std"]:.3f}'
          f'  Cap={ablation[bv]["cap_pct"]:.2f}%  DegPen={ablation[bv]["deg_penalty"]:.5f}')


Running beta ablation: 7 values x 300 eps each...


  beta=  0.0: T_MAE=5.487+/-0.439  Cap=102.15%  DegPen=0.00240


  beta=  0.5: T_MAE=5.511+/-0.367  Cap=102.16%  DegPen=0.00232


  beta=  1.0: T_MAE=5.587+/-0.617  Cap=102.18%  DegPen=0.00229


  beta=  2.0: T_MAE=5.395+/-0.342  Cap=102.13%  DegPen=0.00241


  beta=  5.0: T_MAE=5.384+/-0.438  Cap=102.13%  DegPen=0.00246


  beta= 10.0: T_MAE=5.475+/-0.409  Cap=102.15%  DegPen=0.00241


  beta= 20.0: T_MAE=5.343+/-0.366  Cap=102.11%  DegPen=0.00249


## 8 · Fixed-Profile Trajectory Collection

In [8]:
random.seed(SEED)
fixed_profile=random.choice(profiles)

def run_fixed(ag, profile, beta_val, is_pid=False):
    env_f=BatteryThermalEnvV2([profile],alpha=ALPHA,beta=beta_val,gamma_e=GAMMA_E,steps=STEPS)
    st=env_f.reset()
    if is_pid: ag.reset()
    Ts=[]; Ps=[]; Qs=[]; Rs=[]; DPs=[]
    for _ in range(env_f.steps_per_ep):
        a=ag.act(st,explore=False) if not is_pid else ag.act(st)
        ns,r,done,info=env_f.step(a)
        Ts.append(info['T']); Ps.append(info['P_cool'])
        Qs.append(info['q_pred']); Rs.append(r); DPs.append(info['deg_penalty'])
        st=ns
        if done: break
    Qref=env_f.Q_ep_ref
    return {'T':Ts,'P':Ps,'Q':Qs,'R':Rs,'DP':DPs,'Qref':Qref}

traj_td3 = run_fixed(agent_main, fixed_profile, BETA, is_pid=False)
traj_ab0 = run_fixed(agent_ab0,  fixed_profile, 0.0,  is_pid=False)
traj_pid = run_fixed(pid,         fixed_profile, BETA, is_pid=True)
print(f'Steps collected: {len(traj_td3["T"])} | Q_ref={traj_td3["Qref"]:.4f} Ah')


Steps collected: 300 | Q_ref=1.1031 Ah


## 9 · Publication-Quality Figures

In [9]:
eval_eps=list(range(EVAL_EVERY,EPISODES+1,EVAL_EVERY))
t=np.arange(len(traj_td3['T']))
n=min(len(eval_eps),len(eval_main))
W=20  # smoothing window

# ── Fig 1: Training curves ──────────────────────────────────────────────────
fig,axes=plt.subplots(1,2,figsize=(15,5))
fig.suptitle('Fig 1 — TD3 Training Curves (1500 episodes)',fontsize=13,fontweight='bold')
ax=axes[0]
ax.plot(range(1,len(train_R_main)+1),train_R_main,alpha=0.2,color='steelblue')
ax.plot(range(1,len(train_R_ab0)+1), train_R_ab0, alpha=0.2,color='darkorange')
if len(train_R_main)>=W:
    ma=np.convolve(train_R_main,np.ones(W)/W,'valid')
    mb=np.convolve(train_R_ab0, np.ones(W)/W,'valid')
    ax.plot(range(W,len(train_R_main)+1),ma,'steelblue',lw=2,label=f'TD3-Surrogate (beta={BETA})')
    ax.plot(range(W,len(train_R_ab0)+1), mb,'darkorange',lw=2,label='TD3-NoSurrogate (beta=0)')
ax.set_xlabel('Episode'); ax.set_ylabel('Total Reward')
ax.set_title('Training Reward (20-ep MA)'); ax.legend(); ax.grid(alpha=0.3)
ax=axes[1]
ax.plot(eval_eps[:n],[m['T_MAE'] for m in eval_main[:n]],'b-o',lw=2,ms=5,label=f'TD3-Surrogate')
ax.plot(eval_eps[:n],[m['T_MAE'] for m in eval_ab0[:n]], 'o--',color='darkorange',lw=2,ms=5,label='TD3-NoSurrogate')
if eval_pid_:
    ax.axhline(np.mean([m['T_MAE'] for m in eval_pid_]),color='red',ls='--',lw=2,label='PID')
ax.set_xlabel('Episode'); ax.set_ylabel('Temp MAE (°C)')
ax.set_title('Temp Control Error'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR,'fig1_training.png'),dpi=150,bbox_inches='tight')
plt.show(); print('Saved fig1_training.png')

# ── Fig 2: Trajectory comparison (3 controllers) ────────────────────────────
Qref=traj_td3['Qref']
fig,axes=plt.subplots(2,3,figsize=(18,10))
fig.suptitle('Fig 2 — Trajectories: TD3-Surrogate vs TD3-NoSurrogate vs PID\n(identical current profile)',
             fontsize=13,fontweight='bold')
for i,(traj,nm,clr) in enumerate([
        (traj_td3,f'TD3-Surrogate (b={BETA})','steelblue'),
        (traj_ab0,'TD3-NoSurrogate (b=0)','darkorange'),
        (traj_pid,'PID Baseline','crimson')]):
    t2=np.arange(len(traj['T']))
    ax=axes[0,i]
    ax.plot(t2,traj['T'],color=clr,lw=2)
    ax.axhline(30,color='green',ls='--',lw=1.5,label='Target 30°C')
    ax.axhline(45,color='red',ls='--',lw=1.5,label='Safety 45°C')
    ax.fill_between(t2,28,32,alpha=0.1,color='green')
    ax.set_title(f'{nm}\nT_MAE={np.mean(np.abs(np.array(traj["T"])-30)):.2f}°C')
    ax.set_xlabel('Timestep'); ax.set_ylabel('Temp (°C)')
    ax.legend(fontsize=7); ax.grid(alpha=0.3)
    ax=axes[1,i]
    ax.plot(t2,np.array(traj['Q'])/Qref*100,color=clr,lw=2)
    ax.axhline(100,color='green',ls='--',lw=1)
    ax.set_title(f'Predicted Capacity (% of Q_ref)')
    ax.set_xlabel('Timestep'); ax.set_ylabel('Capacity (%)')
    ax.set_ylim(95,105); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR,'fig2_trajectories.png'),dpi=150,bbox_inches='tight')
plt.show(); print('Saved fig2_trajectories.png')

# ── Fig 3: Beta ablation Pareto curve ───────────────────────────────────────
bvals=sorted(ablation.keys())
t_maes=[ablation[b]['T_MAE']    for b in bvals]
t_stds=[ablation[b]['T_MAE_std'] for b in bvals]
caps  =[ablation[b]['cap_pct']  for b in bvals]
c_stds=[ablation[b]['cap_pct_std'] for b in bvals]
fig,axes=plt.subplots(1,3,figsize=(17,5))
fig.suptitle('Fig 3 — beta Ablation: Degradation Coupling Pareto Trade-off',
             fontsize=13,fontweight='bold')
ax=axes[0]
ax.errorbar(bvals,t_maes,yerr=t_stds,fmt='b-o',lw=2,ms=8,capsize=5)
ax.axvline(BETA,color='red',ls='--',lw=2,label=f'Proposed beta={BETA}')
ax.set_xlabel('beta'); ax.set_ylabel('Temperature MAE (°C)')
ax.set_title('Thermal Accuracy vs beta'); ax.legend(); ax.grid(alpha=0.3)
ax=axes[1]
ax.errorbar(bvals,caps,yerr=c_stds,fmt='g-o',lw=2,ms=8,capsize=5)
ax.axvline(BETA,color='red',ls='--',lw=2,label=f'Proposed beta={BETA}')
ax.axhline(100,color='gray',ls=':',lw=1)
ax.set_xlabel('beta'); ax.set_ylabel('Capacity Preserved (%)')
ax.set_title('Capacity vs beta'); ax.legend(); ax.grid(alpha=0.3)
ax=axes[2]
sc=ax.scatter(t_maes,caps,c=bvals,cmap='plasma',s=200,zorder=5,edgecolors='k')
for b,tm,c,ts,cs in zip(bvals,t_maes,caps,t_stds,c_stds):
    ax.annotate(f'β={b}',(tm,c),xytext=(6,4),textcoords='offset points',fontsize=9,fontweight='bold')
ax.errorbar(t_maes,caps,xerr=t_stds,yerr=c_stds,fmt='none',color='gray',alpha=0.5,capsize=4)
plt.colorbar(sc,ax=ax,label='beta')
ax.set_xlabel('Temperature MAE (°C)'); ax.set_ylabel('Capacity Preserved (%)')
ax.set_title('Pareto Frontier: Thermal Accuracy vs Capacity'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR,'fig3_ablation_pareto.png'),dpi=150,bbox_inches='tight')
plt.show(); print('Saved fig3_ablation_pareto.png')

# ── Fig 4: Degradation penalty over training ─────────────────────────────────
fig,axes=plt.subplots(1,3,figsize=(16,5))
fig.suptitle('Fig 4 — Degradation Penalty Fires Correctly (v2 fix validated)',
             fontsize=13,fontweight='bold')
axes[0].plot(eval_eps[:n],[m['deg_penalty'] for m in eval_main[:n]],'b-o',lw=2,
             label=f'TD3-Surrogate (b={BETA})')
axes[0].plot(eval_eps[:n],[m['deg_penalty'] for m in eval_ab0[:n]],'o--',color='darkorange',lw=2,
             label='TD3-NoSurrogate (b=0)')
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Mean Degradation Penalty (Ah)')
axes[0].set_title('Degradation Penalty (>0 means reward fires)'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(eval_eps[:n],[m['cap_pct'] for m in eval_main[:n]],'b-o',lw=2,label='TD3-Surrogate')
axes[1].plot(eval_eps[:n],[m['cap_pct'] for m in eval_ab0[:n]],'o--',color='darkorange',lw=2,label='TD3-NoSurrogate')
if eval_pid_:
    axes[1].axhline(np.mean([m['cap_pct'] for m in eval_pid_]),color='red',ls='--',lw=2,label='PID')
axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Capacity Preserved (%)')
axes[1].set_title('Capacity Preservation'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[2].plot(eval_eps[:n],[m['cooling'] for m in eval_main[:n]],'b-o',lw=2,label='TD3-Surrogate')
axes[2].plot(eval_eps[:n],[m['cooling'] for m in eval_ab0[:n]],'o--',color='darkorange',lw=2,label='TD3-NoSurrogate')
if eval_pid_:
    axes[2].axhline(np.mean([m['cooling'] for m in eval_pid_]),color='red',ls='--',lw=2,label='PID')
axes[2].set_xlabel('Episode'); axes[2].set_ylabel('Total Cooling Energy (J)')
axes[2].set_title('Cooling Energy Consumption'); axes[2].legend(); axes[2].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR,'fig4_degradation_penalty.png'),dpi=150,bbox_inches='tight')
plt.show(); print('Saved fig4_degradation_penalty.png')


Saved fig1_training.png


Saved fig2_trajectories.png


Saved fig3_ablation_pareto.png
Saved fig4_degradation_penalty.png


## 10 · Final Evaluation (50 Episodes) — Paper Table

In [10]:
print('Running 50-episode final evaluation...')
res={'TD3-Surrogate':[],'TD3-NoSurrogate':[],'PID':[]}
for _ in range(50):
    res['TD3-Surrogate'].append(run_ep(agent_main,env_main,explore=False))
    res['TD3-NoSurrogate'].append(run_ep(agent_ab0, env_ab0, explore=False))
    res['PID'].append(run_ep(pid,env_main,explore=False,is_pid=True))

metrics=['R','T_MAE','cap_pct','deg_penalty','cooling','safety']
labels =['Total Reward','Temp MAE (°C)','Cap Preserved (%)','Deg Penalty (Ah)','Cooling (J)','Safety Violations']

print('='*80)
print('FINAL RESULTS (50-episode mean ± std) — for journal paper')
print('='*80)
hdr=f'{"Metric":<25}  {"TD3-Surrogate":>22}  {"TD3-NoSurrogate":>22}  {"PID":>18}'
print(hdr); print('-'*90)
for k,lbl in zip(metrics,labels):
    row=''
    for name in ['TD3-Surrogate','TD3-NoSurrogate','PID']:
        v=[m[k] for m in res[name]]
        row+=f'{np.mean(v):10.3f}+/-{np.std(v):.3f}    '
    print(f'{lbl:<25}  {row}')

# Save full results JSON
out_json={
    k:{m:{j:float(np.mean([x[j] for x in res[k]])) for j in metrics} for m,_ in [('mean',0)]}
    for k in res
}
out_json['ablation']={str(b):v for b,v in ablation.items()}
out_json['hyperparams']={'ALPHA':ALPHA,'BETA':BETA,'GAMMA_E':GAMMA_E,
                         'EPISODES':EPISODES,'STEPS':STEPS,'algorithm':'TD3'}
with open(os.path.join(RESULTS_DIR,'results_summary_v2.json'),'w') as f:
    json.dump(out_json,f,indent=2)

print(f'\nAll outputs saved to: {RESULTS_DIR}')
print(f'Model: {MODEL_DIR}/thermal_agent_td3_v2.pth')


Running 50-episode final evaluation...


FINAL RESULTS (50-episode mean ± std) — for journal paper
Metric                              TD3-Surrogate         TD3-NoSurrogate                 PID
------------------------------------------------------------------------------------------
Total Reward                 -814.882+/-156.602      -776.346+/-160.027      -978.369+/-132.140    
Temp MAE (°C)                   5.170+/-0.507         5.050+/-0.524         5.681+/-0.399    
Cap Preserved (%)             102.012+/-1.097       102.110+/-1.419       102.155+/-0.977    
Deg Penalty (Ah)                0.002+/-0.004         0.003+/-0.006         0.001+/-0.003    
Cooling (J)                   371.411+/-235.349       350.271+/-512.049      1738.574+/-176.761    
Safety Violations               0.000+/-0.000         0.000+/-0.000         0.000+/-0.000    

All outputs saved to: /Users/mahizhan/Documents/Sem7/Github/data-driven-prediction-of-battery-cycle-life-before-capacity-degradation/thermal_agent_v2_results
Model: /Users/mahizhan